# EXP-009 (hướng B) — Train lại với tiền xử lý chuẩn CLIP
Mọi ảnh (CLIP lúc train, BLIP-2 lúc tạo target, CLIP lúc test) dùng `--preprocess clip` = Resize bicubic 224 + ToTensor + Normalize mean/std CLIP (= P2 của EXP-008).

1. Cache lại **target BLIP-2** (cần A100, ~1–1,5 giờ, chạy tiếp được nếu bị ngắt) và **feature CLIP** với `clip`.
2. Train từ cache, 3 seed × 2 cấu hình: `plain` (như Exp-CLIP gốc; e5 ≈ EXP-000) và `anchor` (crossA, λ=1, ep30 = EXP-004).
3. Test trong notebook (như EXP-008): head mới test bằng P2, head cũ bằng P0; thêm 1 lần `test.py --preprocess clip` để đối chiếu.

**Tiêu chí (chọn trước):** `anchor` e30 @P2 so với EXP-004 @P0: Mean4 UAR (Ensemble Five) ≥ +0.5, không dataset nào < −0.5. Phụ: `plain` e5 @P2 so với EXP-000 @P0 (cùng ngưỡng).
Chẩn đoán: RAFDB của `anchor` @P2 so với EXP-004 @P2 (EXP-008: 55.24). Tăng ≥ +1.0 nghĩa là train lại có giúp.

# Cấu hình

In [1]:
SEEDS = [1, 2, 3]
EPOCHS = 30
SAVE_EPOCHS = [5, 10, 20, 30]
LR = 1e-3
PREPROCESS = "clip"
VARIANTS = {   # tên → cờ thêm cho train.py
    "plain": "",
    "anchor": "--text-bank {bank} --anchor-weight 1 --anchor-type cross",
}
job_of = lambda v, seed: {"plain": f"exp009_clipnorm_plain_l14_seed{seed}",
                          "anchor": f"exp009_clipnorm_anchor_crossA_l1_l14_seed{seed}"}[v]
BRANCH = "exp/009-retrain-clip-normalize"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp009"
PRE_DIR = f"{OUT_DIR}/pre_clip"                                     # targets/chunk_*.npz mới (preprocess clip)
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"
TARGETS_DONE = f"{PRE_DIR}/targets_done.txt"                        # đánh dấu đã cache xong toàn bộ target
OLD_PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"   # cache cũ (raw) để so sánh
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{EXP004_DIR}/text_bank_A_l14.npz"                          # chỉ có text → dùng lại của EXP-004
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"
SCRIPT = "tools/precompute_au_weights.py"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

def hf_offline():
    # FlanT5 đã có trong cache thì không gọi HuggingFace nữa
    os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

# Mount Drive & GPU
Chưa cache xong target thì bắt buộc A100 (BLIP-2 FlanT5-XXL cần ≥ 35GB). Đã xong thì GPU nào cũng được.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(PRE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
NEED_BLIP2 = not os.path.exists(TARGETS_DONE)
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
if NEED_BLIP2:
    assert gpu_mem >= 35000, f"GPU chỉ có {gpu_mem} MiB — cần A100 để cache target BLIP-2 (Runtime > Change runtime type)"
print("Cần BLIP-2:", NEED_BLIP2)

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
Cần BLIP-2: True


# Clone repo & giải nén data

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 454, done.
remote: Counting objects: 100% (454/454), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 454 (delta 333), reused 394 (delta 274), pack-reused 0 (from 0)
Receiving objects: 100% (454/454), 32.30 MiB | 34.13 MiB/s, done.
Resolving deltas: 100% (333/333), done.
/content/Exp-CLIP
Already on 'exp/009-retrain-clip-normalize'
Your branch is up to date with 'origin/exp/009-retrain-clip-normalize'.
Already up to date.
35c1b3f (HEAD -> exp/009-retrain-clip-normalize, origin/exp/009-retrain-clip-normalize) EXP-009: --preprocess flag (CLIP normalisation) for train/test/cache tools
Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Môi trường Python 3.10 (lavis)

In [4]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"
# môi trường Colab mặc định (dùng cho phần test trong notebook)
!pip install -q ftfy "setuptools<81"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Kiểm tra: transform `clip` = bộ xử lý eval của BLIP-2 trong LAVIS
Lệch tối đa phải ~0 (chỉ sai số làm tròn). Lệch lớn thì `image_transform('clip')` không khớp BLIP-2, dừng lại.

In [5]:
check = r"""
import glob, sys
from PIL import Image
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models import load_preprocess
sys.path.insert(0, '.')
from data_loader.image_transform import image_transform
cfg = OmegaConf.load(registry.get_model_class('blip2_t5').default_config_path('pretrain_flant5xxl'))
vis, _ = load_preprocess(cfg.preprocess)
print('LAVIS eval processor:', vis['eval'].transform)
tf = image_transform('clip')
paths = sorted(glob.glob('/content/CAER-S_Face/train/*/*'))[::5000][:10]
d = max((vis['eval'](Image.open(p).convert('RGB')) - tf(Image.open(p).convert('RGB'))).abs().max().item() for p in paths)
print(f'{len(paths)} ảnh | max |LAVIS - clip| = {d:.2e}')
assert d < 1e-4, 'transform clip không khớp BLIP-2'
"""
open("/content/check_tf.py", "w").write(check)
run("/content/venv310/bin/python /content/check_tf.py")

LAVIS eval processor: Compose(
    Resize(size=(224, 224), interpolation=bicubic, max_size=None, antialias=warn)
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
10 ảnh | max |LAVIS - clip| = 0.00e+00


# 1) Cache target BLIP-2 với `--preprocess clip` (A100)
Ghi từng chunk vào Drive → bị ngắt thì chạy lại cell này, nó tiếp từ chỗ dừng.

In [6]:
if NEED_BLIP2:
    hf_offline()
    run(f'/content/venv310/bin/python {SCRIPT} target --train-data {TRAIN_DATA} --out-dir {PRE_DIR} '
        f'--limit 0 --batch-size 64 --workers 8 --instruction "{INSTRUCTION}" --preprocess {PREPROCESS}')
    open(TARGETS_DONE, "w").write("ok")
!ls "$PRE_DIR/targets" | wc -l

0 done, 48938 to encode
/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

  0%|          | 0.00/1.89G [00:00<?, ?B/s]
  0%|          | 48.0k/1.89G [00:00<1:15:15, 449kB/s]
  0%|          | 96.0k/1.89G [00:00<1:14:21, 454kB/s]
  0%|          | 224k/1.89G [00:00<40:49, 827kB/s]   
  0%|          | 464k/1.89G [00:00<23:15, 1.45MB/s]
  0%|          | 944k/1.89G [00:00<12:37, 2.67MB/s]
  0%|          | 1.95M/1.89G [00:00<06:15, 5.39MB/s]
  0%|          | 4.27M/1.89G [00:00<02:56, 11.5MB/s]
  0%|          | 8.48M/1.89G [00:00<01:34, 21.4MB/s]
  1%|          | 12.8M/1.89G [00:00<01:10, 28.7MB/s]
  1%|          | 17.0M/1.89G [00:01<01:00, 33.1MB/s]
  1%|          | 20.2M/1.89G [00:01<01:02, 31.9MB/s]
  1%|▏         | 25.1M/1.89G [00:01<00:53, 37.1MB/s]
  2%|

# 2) Cache feature CLIP với `--preprocess clip` + text bank

In [7]:
if not os.path.exists(CLIP_CACHE):
    run(f'/content/venv310/bin/python tools/cache_clip_features.py --train-data {TRAIN_DATA} --out {CLIP_CACHE} '
        f'--load-model CLIP_L14 --batch-size 256 --workers 8 --preprocess {PREPROCESS}')
if not os.path.exists(BANK):
    run(f'/content/venv310/bin/python tools/build_text_bank.py --variant A --out {BANK} --load-model CLIP_L14')
!ls -lh "$CLIP_CACHE" "$BANK"


  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  1%|▎                                     | 7.48M/890M [00:00<00:11, 77.9MiB/s]
  2%|▋                                     | 14.9M/890M [00:00<00:20, 45.5MiB/s]
  2%|▊                                     | 20.0M/890M [00:00<00:38, 23.5MiB/s]
  3%|▉                                     | 23.2M/890M [00:00<00:37, 24.2MiB/s]
  3%|█                                     | 26.2M/890M [00:01<00:39, 22.9MiB/s]
  4%|█▎                                    | 32.0M/890M [00:01<00:29, 30.3MiB/s]
  4%|█▌                                    | 35.6M/890M [00:01<00:29, 30.5MiB/s]
  5%|█▋                                    | 40.3M/890M [00:01<00:25, 34.9MiB/s]
  5%|█▉                                    | 44.1M/890M [00:01<00:35, 25.2MiB/s]
  5%|██                                    | 48.0M/890M [00:01<00:31, 28.3MiB/s]
  6%|██▍                                   | 56.0M/890M [00:01<00:23, 37.0MiB/s]
  7%|██▌                   

# Chuẩn hoá đổi target / feature bao nhiêu?
cos(mới, cũ) trên cùng ảnh. Gần 1 → chuẩn hoá ít ảnh hưởng tới target của LLM.

In [8]:
import glob, numpy as np
def load_targets(d):
    t = {}
    for c in sorted(glob.glob(f"{d}/targets/chunk_*.npz")):
        z = np.load(c)
        t.update(zip(z["paths"], z["all"].astype(np.float32)))
    return t
new_t, old_t = load_targets(PRE_DIR), load_targets(OLD_PRE_DIR)
common = sorted(set(new_t) & set(old_t))
cos_t = np.array([new_t[p] @ old_t[p] for p in common])   # target đã L2-normalize
print(f"Target: {len(new_t)} mới, {len(old_t)} cũ, {len(common)} chung | cos mean {cos_t.mean():.4f}, min {cos_t.min():.4f}, p5 {np.percentile(cos_t, 5):.4f}")
# so với độ khác nhau giữa các ảnh: cos giữa target của 2 ảnh ngẫu nhiên
rng = np.random.default_rng(0); a, b = rng.choice(len(common), (2, 2000))
print(f"  (tham chiếu: cos giữa 2 ảnh khác nhau, target cũ: {np.mean([old_t[common[i]] @ old_t[common[j]] for i, j in zip(a, b)]):.4f})")

nc, oc = np.load(CLIP_CACHE), np.load(f"{OLD_PRE_DIR}/clip_feats_l14.npz")
assert list(nc["paths"]) == list(oc["paths"])
f1, f0 = nc["feats"].astype(np.float32), oc["feats"].astype(np.float32)
cos_f = (f1 * f0).sum(1) / np.linalg.norm(f1, axis=1) / np.linalg.norm(f0, axis=1)
print(f"Feature CLIP: cos mean {cos_f.mean():.4f}, min {cos_f.min():.4f}")
del new_t, old_t, f1, f0

Target: 48938 mới, 48938 cũ, 48938 chung | cos mean 0.9731, min 0.8349, p5 0.9506
  (tham chiếu: cos giữa 2 ảnh khác nhau, target cũ: 0.9548)
Feature CLIP: cos mean 0.8314, min 0.5391


# 3) Train từ cache: 2 cấu hình × 3 seed

In [9]:
for seed in SEEDS:
  for v, extra in VARIANTS.items():
    job = job_of(v, seed)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"{v} seed {seed}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== {v}, seed {seed}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {seed} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 --preprocess {PREPROCESS} '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets {extra.format(bank=BANK)} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== plain, seed 1
Training date:  26-10-22 16:57
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 1
job_id = exp009_clipnorm_plain_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp009
train_data = /data/EECS-IoannisLab/datasets/Static_FER_Datasets/CAERS_Face/train
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp009/pre_clip/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp009/pre_clip/targets
text_bank = None
anchor_weight = 1.0
anchor_type = cross
sep_head = none
head_tie = 0.0
save_epochs = [5, 10, 20, 30]
resume = None
preprocess = clip
************************
Cached t

In [10]:
import re
for seed in SEEDS:
  for v in VARIANTS:
    lines = open(f"{OUT_DIR}/{job_of(v, seed)}-log.txt").read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    print(f"{v} seed {seed}: train acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)},
          "| anchor", [round(a, 3) for a in anc[::max(1, len(anc) // 6)]])

plain seed 1: train acc {5: 45.33, 10: 65.62, 20: 79.25, 30: 84.59} | anchor []
anchor seed 1: train acc {5: 43.1, 10: 64.6, 20: 79.13, 30: 84.71} | anchor [0.199, 0.083, 0.065, 0.061, 0.055, 0.056]
plain seed 2: train acc {5: 45.85, 10: 66.06, 20: 79.14, 30: 84.62} | anchor []
anchor seed 2: train acc {5: 43.86, 10: 65.04, 20: 78.86, 30: 84.56} | anchor [0.297, 0.085, 0.062, 0.057, 0.056, 0.054]
plain seed 3: train acc {5: 45.29, 10: 65.83, 20: 79.15, 30: 84.38} | anchor []
anchor seed 3: train acc {5: 43.29, 10: 64.68, 20: 78.89, 30: 84.58} | anchor [0.293, 0.086, 0.064, 0.057, 0.056, 0.056]


# 4) Đối chiếu bằng `test.py --preprocess clip` (anchor seed 1 e30)

In [11]:
CHECK_JOB = f"{job_of('anchor', 1)}-e30"
!mkdir -p checkpoint log
if not os.path.exists(f"{OUT_DIR}/{CHECK_JOB}-test.txt"):
    !cp "$OUT_DIR/$CHECK_JOB-model.pth" checkpoint/
    run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {CHECK_JOB} --preprocess {PREPROCESS} '
        f'--data-root /content/static_test_sets 2>&1 | tee log/{CHECK_JOB}-test.txt')
    !cp log/$CHECK_JOB-test.txt "$OUT_DIR/"
txt_log = open(f"{OUT_DIR}/{CHECK_JOB}-test.txt").read()
print(txt_log[-1500:])

************************
load_model = CLIP_L14
job_id = exp009_clipnorm_anchor_crossA_l1_l14_seed1-e30
data_root = /content/static_test_sets
preprocess = clip
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 53.54/55.08
************************* AffectNet7
UAR/WAR: 42.71/42.71
************************* AffectNet8
UAR/WAR: 36.52/36.52
************************* FERPlus
UAR/WAR: 46.95/45.03
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 53.67/56.84
************************* AffectNet7
UAR/WAR: 45.14/45.14
************************* AffectNet8
UAR/WAR: 37.92/37.92
************************* FERPlus
UAR/WAR:

# 5) Test trong notebook
Feature ảnh test P0/P2 × (gốc, lật) tính 1 lần; mỗi head chỉ là phép nhân ma trận (như EXP-008).

In [12]:
import sys, glob, re, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
TRANSFORMS = {"P0": image_transform("raw"), "P2": image_transform("clip")}
CFG_NAMES = ["P0", "P0+flip", "P2", "P2+flip"]

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {   # tên → ({seed: checkpoint}, tiền xử lý khớp với lúc train)
    "CLIP gốc": ({0: None}, "P0"),
    "Paper": ({0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"}, "P0"),
    "EXP-000": ({s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS}, "P0"),
    "EXP-004": ({s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS}, "P0"),
    "EXP-009 plain e5": ({s: f"{OUT_DIR}/{job_of('plain', s)}-e5-model.pth" for s in SEEDS}, "P2"),
    "EXP-009 plain e30": ({s: f"{OUT_DIR}/{job_of('plain', s)}-e30-model.pth" for s in SEEDS}, "P2"),
    "EXP-009 anchor e30": ({s: f"{OUT_DIR}/{job_of('anchor', s)}-e30-model.pth" for s in SEEDS}, "P2"),
}
for m, (d, _) in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[(ds, prep)] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        for prep, tf in TRANSFORMS.items():
            ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", tf)
            loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
            f0, f1 = [], []
            for x, _ in loader:
                x = x.cuda()
                f0.append(clip_model.encode_image(x).float().cpu())
                f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
            img[(ds_name, prep)] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[(ds_name, "P0")][0].shape), classes[ds_name])

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']


In [13]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, ds_name, cfg, pn):
    f0, f1 = img[(ds_name, cfg[:2])]
    i, t = ((f0 + f1) / 2 if cfg.endswith("+flip") else f0), txt[(ds_name, pn)]   # TTA: trung bình trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, (d, _) in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    r = metrics(y[ds_name], predict(W, ds_name, cfg, pn), N_CLS[ds_name])
                    rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn,
                                 "UAR": r["UAR"], "WAR": r["WAR"],
                                 **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp009_retrain_clipnorm-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

544 dòng → /content/drive/MyDrive/ExpCLIP_output/exp009


# Kiểm tra pipeline
(a) Head cũ @P0 phải khớp số đã ghi (`test.py`). (b) Head mới @P2 phải khớp `test.py --preprocess clip` ở bước 4. Lệch ≤ ~0.4 là do fp16.

In [14]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004|1|RAFDB": [58.73, 65.94], "EXP-004|1|AffectNet7": [45.83, 45.83], "EXP-004|1|AffectNet8": [39.6, 39.6], "EXP-004|1|FERPlus": [46.14, 52.03], "EXP-004|2|RAFDB": [57.07, 64.77], "EXP-004|2|AffectNet7": [43.97, 43.97], "EXP-004|2|AffectNet8": [38.75, 38.75], "EXP-004|2|FERPlus": [45.63, 52.45], "EXP-004|3|RAFDB": [58.26, 65.06], "EXP-004|3|AffectNet7": [45.46, 45.46], "EXP-004|3|AffectNet8": [39.07, 39.08], "EXP-004|3|FERPlus": [45.34, 51.44]}

# (b) đọc UAR/WAR Ensemble Five từ log test.py (khối "Prompt Type: ..." → các dòng "<dataset> ... UAR/WAR: x/y")
blk = re.split(r"Prompt Type:\s+", txt_log)
blk = next(b for b in blk if b.startswith(BASE_PROMPT + "\n"))
for ds_name in DATASETS:
    mm = re.search(rf"\* {ds_name}\nUAR/WAR: ([\d.]+)/([\d.]+)", blk)
    if mm:
        REF[f"EXP-009 anchor e30|1|{ds_name}|P2"] = [float(mm.group(1)), float(mm.group(2))]
print("Đọc từ test.py:", {k: v for k, v in REF.items() if k.endswith("|P2")})

chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name, *cfg = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == (cfg[0] if cfg else "P0")) & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))

Đọc từ test.py: {'EXP-009 anchor e30|1|RAFDB|P2': [54.38, 56.84], 'EXP-009 anchor e30|1|AffectNet7|P2': [44.31, 44.31], 'EXP-009 anchor e30|1|AffectNet8|P2': [37.97, 37.98], 'EXP-009 anchor e30|1|FERPlus|P2': [48.34, 50.43]}
28/28 cấu hình so được | max |ΔUAR| = 0.35, max |ΔWAR| = 0.08
                            key  dUAR  dWAR
              EXP-000|3|FERPlus -0.35  0.06
                EXP-000|2|RAFDB  0.09 -0.00
           EXP-004|1|AffectNet8 -0.08 -0.08
           EXP-000|2|AffectNet7 -0.06 -0.06
           EXP-004|2|AffectNet7  0.06  0.06
EXP-009 anchor e30|1|FERPlus|P2  0.06  0.06


# Bảng 1 — UAR, mỗi head với tiền xử lý khớp lúc train (Ensemble Five, mean ± std qua seed)

In [15]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

u = res.pivot_table(index=["model", "seed", "prompt", "config"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u.mean(1)

def get(m, cfg, pn=BASE_PROMPT):
    return u.loc[m].xs(pn, level="prompt").xs(cfg, level="config")   # index: seed

for pn in PROMPT_NAMES:
    t = pd.DataFrame({f"{m} @{c}{f}": get(m, c + f, pn).agg(fmt) for m, (_, c) in HEADS.items() for f in ["", "+flip"]}).T
    print(f"--- {pn}\n{t.to_string()}\n")

--- Expression Ensemble Five
dataset                             RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
CLIP gốc @P0                        52.07         40.74         36.42         38.97         42.05
CLIP gốc @P0+flip                   51.90         41.26         36.78         38.73         42.17
Paper @P0                           57.90         44.23         38.95         43.17         46.06
Paper @P0+flip                      59.12         44.54         39.40         43.31         46.59
EXP-000 @P0                  58.10 ± 0.69  44.14 ± 0.55  38.82 ± 0.60  41.42 ± 1.02  45.62 ± 0.40
EXP-000 @P0+flip             59.39 ± 0.95  44.64 ± 0.48  39.06 ± 0.71  41.62 ± 0.83  46.18 ± 0.62
EXP-004 @P0                  58.02 ± 0.81  45.09 ± 0.93  39.13 ± 0.38  45.72 ± 0.41  46.99 ± 0.58
EXP-004 @P0+flip             58.94 ± 0.34  45.49 ± 0.98  39.49 ± 0.57  46.46 ± 0.68  47.59 ± 0.51
EXP-009 plain e5 @P2         54.35 ± 4.72  43.58 ± 0.48  37.13 ± 0.38  43.62 ± 0.24  44.6

# Bảng 2 — Tiêu chí (chênh lệch theo cặp seed, Ensemble Five)

In [16]:
def compare(a, b, pn=BASE_PROMPT):
    d = get(*a, pn) - get(*b, pn)
    return d.agg(fmt), d.mean()

checks = {
    "CHÍNH: anchor e30 @P2 − EXP-004 @P0": (("EXP-009 anchor e30", "P2"), ("EXP-004", "P0")),
    "PHỤ: plain e5 @P2 − EXP-000 @P0": (("EXP-009 plain e5", "P2"), ("EXP-000", "P0")),
    "plain e30 @P2 − EXP-000 @P0": (("EXP-009 plain e30", "P2"), ("EXP-000", "P0")),
    "Chẩn đoán: anchor e30 @P2 − EXP-004 @P2": (("EXP-009 anchor e30", "P2"), ("EXP-004", "P2")),
    "Chẩn đoán: plain e5 @P2 − EXP-000 @P2": (("EXP-009 plain e5", "P2"), ("EXP-000", "P2")),
    "+flip: anchor e30 @P2+flip − EXP-004 @P0+flip": (("EXP-009 anchor e30", "P2+flip"), ("EXP-004", "P0+flip")),
}
t, verdict = {}, {}
for name, (a, b) in checks.items():
    t[name], mean = compare(a, b)
    if name.startswith(("CHÍNH", "PHỤ")):
        ok = mean["Mean4"] >= 0.5 and (mean[DATASETS] >= -0.5).all()
        verdict[name] = f"Δ Mean4 {mean['Mean4']:+.2f}, min Δ dataset {mean[DATASETS].min():+.2f} → {'ĐẠT' if ok else 'KHÔNG đạt'}"
print(pd.DataFrame(t).T.to_string(), "\n")
for k, v in verdict.items():
    print(f"{k}: {v}")

dataset                                               RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
CHÍNH: anchor e30 @P2 − EXP-004 @P0            -6.02 ± 2.78  -0.73 ± 0.89  -1.13 ± 0.62   1.05 ± 1.05  -1.71 ± 0.89
PHỤ: plain e5 @P2 − EXP-000 @P0                -3.75 ± 4.16  -0.56 ± 1.00  -1.68 ± 0.80   2.20 ± 1.02  -0.95 ± 1.67
plain e30 @P2 − EXP-000 @P0                    -5.17 ± 2.38   0.36 ± 0.20  -1.62 ± 0.33   5.18 ± 1.37  -0.31 ± 0.98
Chẩn đoán: anchor e30 @P2 − EXP-004 @P2        -3.30 ± 3.56   0.58 ± 0.89  -1.17 ± 0.79  -3.24 ± 1.18  -1.79 ± 1.57
Chẩn đoán: plain e5 @P2 − EXP-000 @P2          -0.66 ± 4.94  -0.02 ± 0.75  -1.98 ± 0.56  -5.79 ± 1.57  -2.11 ± 1.48
+flip: anchor e30 @P2+flip − EXP-004 @P0+flip  -5.10 ± 2.73  -1.24 ± 1.20  -1.49 ± 0.91   0.24 ± 1.98  -1.90 ± 1.25 

CHÍNH: anchor e30 @P2 − EXP-004 @P0: Δ Mean4 -1.71, min Δ dataset -6.02 → KHÔNG đạt
PHỤ: plain e5 @P2 − EXP-000 @P0: Δ Mean4 -0.95, min Δ dataset -3.75 → KHÔNG đạt


# Bảng 3 — Accuracy từng lớp (Ensemble Five, mean seed)
Neutral có còn sụp khi train lại với `clip` không?

In [17]:
for ds_name in ["RAFDB", "FERPlus"]:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    t = {}
    for m, c in [("EXP-004", "P0"), ("EXP-004", "P2"), ("EXP-009 anchor e30", "P2"), ("EXP-000", "P0"), ("EXP-009 plain e5", "P2")]:
        r = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT) & (res.config == c)]
        t[f"{m} @{c}"] = r[cols].mean().values
    t = pd.DataFrame(t, index=classes[ds_name]).T
    print(f"--- {ds_name}\n{t.round(1).to_string()}\n")

--- RAFDB
                        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
EXP-004 @P0                  89.3     52.6     32.9   91.4      70.2     42.7  27.0
EXP-004 @P2                  88.6     46.2      7.7   93.6      71.2     46.0  33.8
EXP-009 anchor e30 @P2       69.3     53.8     33.0   97.7      65.0     13.5  31.5
EXP-000 @P0                  87.2     45.7     45.4   88.9      60.3     50.0  29.3
EXP-009 plain e5 @P2         72.3     50.0     56.1   89.9      71.2     24.2  16.7

--- FERPlus
                        Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear  Contempt
EXP-004 @P0                  85.3     44.9     22.7   57.7      80.1     27.0  12.6      35.6
EXP-004 @P2                  78.9     50.9      7.3   73.4      87.6     42.9  29.3      30.0
EXP-009 anchor e30 @P2       63.5     59.8     22.9   66.5      81.2      9.5  24.1      46.7
EXP-000 @P0                  88.7     35.1     34.9   43.6      70.7     23.8  11.2      23.3
EXP

# Lưu JSON (cùng định dạng `research_log/results/`)

In [18]:
out = {"source": "notebooks/EXP_009_retrain_clipnorm.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-009-retrain-clipnorm.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-009-retrain-clipnorm.json")

544 dòng → /content/drive/MyDrive/ExpCLIP_output/exp009/EXP-009-retrain-clipnorm.json
